# 1. Carregamento da base
Leitura do arquivo Excel e visualizacao inicial do DataFrame.

In [30]:
import pandas as pd

arquivo = "MLCQCodeSmellSamples.xlsx"
df = pd.read_excel(arquivo)

print(df.shape)
df.head()

(14739, 15)


,id,reviewer_id,sample_id,smell,severity,review_timestamp,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project
0,526,6,5771277,feature envy,none,2019-03-27 10:34:53.041496,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1.0
1,527,6,5771277,long method,none,2019-03-27 10:34:53.042443,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1.0
2,528,6,5786929,blob,critical,2019-03-27 10:37:38.107923,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1.0
3,529,6,5786929,data class,critical,2019-03-27 10:37:38.109068,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1.0
4,530,6,5788107,feature envy,none,2019-03-27 10:37:49.627100,function,org.apache.tika.parser.ocr.TesseractOCRConfig#...,git@github.com:apache/tika.git,4131c6e30f2e0eb1feb85e0f7576531d4e830468,/tika-parsers/src/main/java/org/apache/tika/pa...,531,534,https://github.com/apache/tika/blob/4131c6e30f...,1.0


## 1.1 Conferencia das colunas
Lista os nomes de colunas para validar a estrutura da base.

In [8]:
df.columns.tolist()

['id',
 'reviewer_id',
 'sample_id',
 'smell',
 'severity',
 'review_timestamp',
 'type',
 'code_name',
 'repository',
 'commit_hash',
 'path',
 'start_line',
 'end_line',
 'link',
 'is_from_industry_relevant_project']

## 1.2 Tamanho antes da limpeza
Mostra quantidade de linhas e colunas antes da deduplicacao.

In [31]:
print(df.shape)

(14739, 15)


## 1.3 Remocao de duplicatas
Remove registros com link repetido para evitar verificacoes redundantes.

In [10]:
df = df.drop_duplicates(subset=["link"])
print(df.shape)

(4770, 15)


# 2. Validacao de links HTTP
Importa bibliotecas para requisicoes e processamento paralelo.

In [11]:
import requests
from requests.exceptions import RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed

## 2.1 Funcao de verificacao individual
Define uma rotina para consultar URL e retornar status HTTP.

In [12]:
def verificar_link(url, timeout=8):
    try:
        response = requests.get(url, timeout=timeout, allow_redirects=True)
        return response.status_code
    except RequestException:
        return None

## 2.2 Funcao de verificacao em lote
Processa varias URLs em paralelo e registra progresso da execucao.

In [5]:
def verificar_links_em_lote(urls, max_workers=10):
    resultados = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {executor.submit(verificar_link, url): url for url in urls}
        
        total = len(futuros)
        concluidos = 0
        
        for futuro in as_completed(futuros):
            url = futuros[futuro]
            try:
                resultados[url] = futuro.result()
            except Exception:
                resultados[url] = None
            
            concluidos += 1
            if concluidos % 100 == 0 or concluidos == total:
                print(f"{concluidos}/{total} links verificados")
    
    return resultados

## 2.3 Padronizacao dos links
Normaliza campo de link e gera lista unica para validacao.

In [13]:
df.columns = df.columns.str.strip()

LINK_COL = "link"

df[LINK_COL] = df[LINK_COL].astype(str).str.strip()
df = df[df[LINK_COL].notna()]
df = df[df[LINK_COL] != ""]
df = df[df[LINK_COL].str.lower() != "nan"]

urls_unicas = df[LINK_COL].unique().tolist()
print(f"Total de links únicos: {len(urls_unicas)}")

Total de links únicos: 4770


## 2.4 Execucao da verificacao
Executa a checagem das URLs e salva os resultados por link.

In [14]:
resultados = verificar_links_em_lote(urls_unicas, max_workers=10)

100/4770 links verificados
200/4770 links verificados
300/4770 links verificados
400/4770 links verificados
500/4770 links verificados
600/4770 links verificados
700/4770 links verificados
800/4770 links verificados
900/4770 links verificados
1000/4770 links verificados
1100/4770 links verificados
1200/4770 links verificados
1300/4770 links verificados
1400/4770 links verificados
1500/4770 links verificados
1600/4770 links verificados
1700/4770 links verificados
1800/4770 links verificados
1900/4770 links verificados
2000/4770 links verificados
2100/4770 links verificados
2200/4770 links verificados
2300/4770 links verificados
2400/4770 links verificados
2500/4770 links verificados
2600/4770 links verificados
2700/4770 links verificados
2800/4770 links verificados
2900/4770 links verificados
3000/4770 links verificados
3100/4770 links verificados
3200/4770 links verificados
3300/4770 links verificados
3400/4770 links verificados
3500/4770 links verificados
3600/4770 links verificados
3

## 2.5 Mapeamento de status
Associa status HTTP no DataFrame e cria indicador de link valido.

In [15]:
df["status_code"] = df[LINK_COL].map(resultados)
df["link_ok"] = df["status_code"] == 200

## 2.6 Diagnostico inicial
Mostra a distribuicao dos codigos HTTP obtidos na primeira rodada.

In [17]:
df["status_code"].value_counts(dropna=False)

status_code
200.0    3740
429.0     674
404.0     351
502.0       4
NaN         1
Name: count, dtype: int64

## 2.7 Coleta de links 429
Separa as URLs com status 429 para reprocessamento controlado.

In [18]:
urls_429 = df.loc[df["status_code"] == 429, LINK_COL].dropna().unique().tolist()
print(f"Links com 429: {len(urls_429)}")

Links com 429: 674


## 2.8 Funcao de retry
Implementa tentativas com espera progressiva para reduzir falhas temporarias.

In [20]:
import time
import requests
from requests.exceptions import RequestException

def verificar_link_com_retry(url, timeout=8, tentativas=3, espera=3):
    for tentativa in range(1, tentativas + 1):
        try:
            response = requests.get(url, timeout=timeout, allow_redirects=True)
            status = response.status_code
            
            if status != 429:
                return status
            
            if tentativa < tentativas:
                time.sleep(espera * tentativa)
                
        except RequestException:
            if tentativa < tentativas:
                time.sleep(espera * tentativa)
            else:
                return None
    
    return 429

## 2.9 Reprocessamento em lote
Prepara rotina paralela para revalidar links que retornaram 429.

In [21]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def verificar_links_reprocessamento(urls, max_workers=3):
    resultados = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {executor.submit(verificar_link_com_retry, url): url for url in urls}
        
        total = len(futuros)
        concluidos = 0
        
        for futuro in as_completed(futuros):
            url = futuros[futuro]
            try:
                resultados[url] = futuro.result()
            except Exception:
                resultados[url] = None
            
            concluidos += 1
            if concluidos % 50 == 0 or concluidos == total:
                print(f"{concluidos}/{total} links 429 reprocessados")
    
    return resultados

## 2.10 Execucao do retry
Executa a segunda rodada de verificacao somente para links 429.

In [22]:
resultados_429 = verificar_links_reprocessamento(urls_429, max_workers=1)

50/674 links 429 reprocessados
100/674 links 429 reprocessados
150/674 links 429 reprocessados
200/674 links 429 reprocessados
250/674 links 429 reprocessados
300/674 links 429 reprocessados
350/674 links 429 reprocessados
400/674 links 429 reprocessados
450/674 links 429 reprocessados
500/674 links 429 reprocessados
550/674 links 429 reprocessados
600/674 links 429 reprocessados
650/674 links 429 reprocessados
674/674 links 429 reprocessados


## 2.11 Atualizacao dos resultados
Aplica os novos status no DataFrame e recalcula a coluna link_ok.

In [23]:
for url, novo_status in resultados_429.items():
    df.loc[df[LINK_COL] == url, "status_code"] = novo_status

df["link_ok"] = df["status_code"] == 200

## 2.12 Diagnostico final
Reconta os codigos HTTP apos o reprocessamento dos erros 429.

In [24]:
df["status_code"].value_counts(dropna=False)

status_code
200.0    4359
404.0     406
502.0       4
NaN         1
Name: count, dtype: int64

# 3. Geracao dos artefatos finais
Filtra os registros validos e exporta os arquivos de saida.

In [28]:
df_limpo = df[df["status_code"] == 200].copy()
print(df_limpo.shape)

(4359, 17)


## 3.1 Exportacao dos arquivos
Salva a base completa com status e a base limpa com status 200.

In [29]:
df.to_excel("MLCQ_status_final.xlsx", index=False)
df_limpo.to_excel("MLCQ_limpo_200.xlsx", index=False)

## 3.2 Area para analises extras
Celula reservada para metricas, graficos ou validacoes adicionais.

## 3.3 Proximos passos
Espaco para documentar melhorias e novas etapas do pipeline.